# Adatok felderítése, pipeline kialakítása

1: SZÜKSÉGES CSOMAGOK IMPORTÁLÁSA

In [30]:
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd

kelvin = 273.15

2: Data exploratory

In [31]:
files = sorted(Path(r"tigge_data").glob("*.grib"))
print(f"- Összesen {len(files)} fájlt tudtunk letölteni")

# kiválasztunk egy mintát
ds = xr.open_dataset(files[0], engine="cfgrib")

print()
print("- Nézzünk bele egy grib fájlba:")
print(ds)

- Összesen 158 fájlt tudtunk letölteni

- Nézzünk bele egy grib fájlba:
<xarray.Dataset> Size: 195kB
Dimensions:            (time: 31, step: 4, values: 97)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    latitude           (values) float64 776B ...
    longitude          (values) float64 776B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    surface            float64 8B ...
Dimensions without coordinates: values
Data variables:
    mx2t6              (time, step, values) float32 48kB ...
    mn2t6              (time, step, values) float32 48kB ...
    t2m                (time, step, values) float32 48kB ...
    tp                 (time, step, values) float32 48kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  E

In [32]:
cities = {
    "Budapest": (47.4979, 19.0402),
    "Debrecen": (47.5316, 21.6273),
    "Miskolc":  (48.1035, 20.7784),
    "Pécs":     (46.0727, 18.2323),
    "Szeged":   (46.2530, 20.1414),
}

In [33]:
print(ds.latitude.values)

[48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736

In [34]:
def nearest_point(ds, cel_latitude, cel_longitude):
    lat = ds.latitude.values
    lon = ds.longitude.values
    distance = (lat - cel_latitude)**2 + (lon - cel_longitude)**2
    idx = np.argmin(distance)
    return ds.isel(values=idx)

In [35]:
print(nearest_point(ds,cities["Budapest"][0],cities["Budapest"][1]))

<xarray.Dataset> Size: 3kB
Dimensions:            (time: 31, step: 4)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    latitude           float64 8B 47.42
    longitude          float64 8B 18.75
    surface            float64 8B ...
Data variables:
    mx2t6              (time, step) float32 496B ...
    mn2t6              (time, step) float32 496B ...
    t2m                (time, step) float32 496B ...
    tp                 (time, step) float32 496B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Ra

In [36]:
print(ds.step.values)

[21600000000000 43200000000000 64800000000000 86400000000000]


3: Egy hónap dataset-jének kialakítása

In [37]:
def daily_aggregation(pont_ds):
    """pont_ds: egy adott város rácspontjára kivágott Dataset (time, step dimenziókkal)"""
    
    daily_max  = pont_ds["mx2t6"].max(dim="step")
    daily_min  = pont_ds["mn2t6"].min(dim="step")
    daily_mean_maxmin = (daily_max + daily_min) / 2
    daily_mean_t2m    = pont_ds["t2m"].mean(dim="step")
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    daily_precipitaion_m  = pont_ds["tp"].isel(step=-1)
    daily_precipitation_mm = daily_precipitaion_m * 1000
    
    df = pd.DataFrame({
        "datum": pont_ds["time"].values,
        "tmax": daily_max.values,
        "tmin": daily_min.values,
        "tkozep_maxmin": daily_mean_maxmin.values,
        "tkozep_t2m": daily_mean_t2m.values,
        "csapadek_mm": daily_precipitation_mm.values,
    })
    return df

In [38]:
budapest_point = nearest_point(ds, cities["Debrecen"][0], cities["Debrecen"][1])
df = daily_aggregation(budapest_point)

In [39]:
print(df.head(10))

       datum        tmax        tmin  tkozep_maxmin  tkozep_t2m   csapadek_mm
0 2006-10-01  296.980042  283.773560     290.376801  290.484802  2.670287e+01
1 2006-10-02  297.306915  287.096771     292.201843  292.253845  1.449585e+02
2 2006-10-03  298.263824  290.910461     294.587158  294.739990  1.678467e+02
3 2006-10-04  298.501160  288.928802     293.714966  293.161835  1.308441e+04
4 2006-10-05  292.285370  286.953918     289.619629  289.096558  2.975464e+02
5 2006-10-06  291.941681  281.516632     286.729156  285.572632 -1.338483e-08
6 2006-10-07  291.806976  278.859863     285.333435  284.664154 -4.280513e-13
7 2006-10-08  291.584961  279.952454     285.768707  285.210083 -4.272358e-13
8 2006-10-09  291.477936  279.867218     285.672577  284.301941 -2.466796e-07
9 2006-10-10  292.309235  279.480865     285.895050  284.657898 -1.340083e-06


Látjuk hogy kicsit furák az adatok. A hőmérséklet Kelvinben van megadva, a csapadéknál pedig apró, numerikus zajokat figyelhetünk meg.
Próbaljuk ezt megoldani!

In [40]:
def daily_aggregation(pont_ds):

    daily_max  = pont_ds["mx2t6"].max(dim="step") - kelvin
    daily_min  = pont_ds["mn2t6"].min(dim="step") - kelvin
    daily_mean_maxmin = (daily_max + daily_min) / 2
    daily_mean_t2m    = pont_ds["t2m"].mean(dim="step") - kelvin
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    daily_precipitaion_m  = pont_ds["tp"].isel(step=-1)
    daily_precipitation_mm = (daily_precipitaion_m * 1000).clip(min = 0)
    
    df = pd.DataFrame({
            "datum": pont_ds["time"].values,
            "d_ta_maxmin": daily_mean_maxmin.values,
            "d_ta_t2m": daily_mean_t2m.values,
            "d_tx": daily_max.values,
            "d_tn": daily_min.values,
            "csapadek_mm": daily_precipitation_mm.values,
        })
    return df

In [41]:
df2 = daily_aggregation(budapest_point)
print(df2)

        datum  d_ta_maxmin   d_ta_t2m       d_tx       d_tn   csapadek_mm
0  2006-10-01    17.226807  17.334808  23.830048  10.623566     26.702871
1  2006-10-02    19.051849  19.103851  24.156921  13.946777    144.958496
2  2006-10-03    21.437149  21.589996  25.113831  17.760468    167.846680
3  2006-10-04    20.564987  20.011841  25.351166  15.778809  13084.412109
4  2006-10-05    16.469650  15.946564  19.135376  13.803925    297.546387
5  2006-10-06    13.579163  12.422638  18.791687   8.366638      0.000000
6  2006-10-07    12.183426  11.514160  18.656982   5.709869      0.000000
7  2006-10-08    12.618713  12.060089  18.434967   6.802460      0.000000
8  2006-10-09    12.522583  11.151947  18.327942   6.717224      0.000000
9  2006-10-10    12.745056  11.507904  19.159241   6.330872      0.000000
10 2006-10-11    12.554611  11.461212  19.438416   5.670807      0.000000
11 2006-10-12    13.028137  11.662537  19.605072   6.451202      0.000000
12 2006-10-13    13.270065  12.549896 

Na, máris jófelé tartunk. Viszont találtam a met.hu-n egy leírást ami kicsit árnyalja a helyzetünket.
[met.hu](https://met.hu/eghajlat/magyarorszag_eghajlata/eghajlati_adatsorok/Budapest/leirasok/szamitasok/)

3: Met.hu szerinti mérési szabványok implementálása és elemzése

A met.hu napi számítási procedúrái kicsit bonyolultabbak. A napi felbontásnál az adott nap értékeinek kiszámításáoz szükségünk van az azt megelőző és az utána levő nap adataira is. Ez értelemszerűen gondot okoz a hónapok utolsó és első napjainak számításánál mert az adatok hónapos felbontásban vannak. Ezért a grib fájlokat úgy kell megnyitni, hogy két szomszédos hónapot összefésülünk.

A dátum-intervallumunk legelső és legutolsó napját viszont nem tudjuk így lekezelni, ezt el kell fogadnunk.

In [51]:
# Napi max és min hőmérséklet

def daily_max_min_methu(pont_ds, day_index):

    prev_day_24h_max = pont_ds["mx2t6"].isel(time = day_index - 1, step = 3)
    current_day_6_12_24_max = pont_ds["mx2t6"].isel(time = day_index, step=[0, 1, 2])
    current_day_max = current_day_6_12_24_max.max()

    daily_max = max(float(prev_day_24h_max), float(current_day_max)) - kelvin


    prev_day_24h_min = pont_ds["mn2t6"].isel(time = day_index - 1, step = 3)
    current_day_6_12_24_min = pont_ds["mn2t6"].isel(time=day_index, step = [0, 1, 2])
    current_day_min = current_day_6_12_24_min.min()

    daily_min = min(float(prev_day_24h_min), float(current_day_min)) - kelvin

    return [daily_max, daily_min]

In [53]:
print(daily_max_min_methu(budapest_point,1))

[24.156915283203148, 13.946771240234398]


In [55]:
# Napi csapadék
def daily_precipitation_methu(pont_ds, day_index):

    current_step6 = float(pont_ds["tp"].isel(time = day_index, step = 0))
    current_step24 = float(pont_ds["tp"].isel(time = day_index, step = 3))
    next_step6 = float(pont_ds["tp"].isel(time = day_index + 1, step = 0))

    precipitation_m = (current_step24 - current_step6) + next_step6

    return max(precipitation_m * 1000, 0)


In [64]:
print(daily_precipitation_methu(budapest_point, 8))

1.205731377617897e-06
